In [47]:
import pandas as pd
import numpy as np
from collections import defaultdict, deque

In [48]:
df = pd.read_csv("../data/processed/bundesliga_2010_2026.csv")

df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

df = df.sort_values("Date").reset_index(drop=True)

print(df.shape)

(4896, 195)


## Recent Form

Die aktuelle Leistungsform eines Teams kann einen wichtigen
Einfluss auf den Ausgang eines Spiels haben.

Für jedes Team speichern wir deshalb die letzten fünf absolvierten
Spiele und berechnen daraus Durchschnittswerte für Punkte, Tore und
Gegentore.

In [ ]:
team_history = defaultdict(lambda: deque(maxlen=5))


## Home and Away Form

Die Gesamtform eines Teams unterscheidet nicht zwischen Heim- und
Auswärtsspielen.

Da Heimvorteil im Fußball relevant ist, betrachten wir zusätzlich
die letzten fünf Heimspiele des Heimteams und die letzten fünf
Auswärtsspiele des Auswärtsteams.

In [ ]:
home_history = defaultdict(lambda: deque(maxlen=5))
away_history = defaultdict(lambda: deque(maxlen=5))

In [50]:
def get_team_stats(history):
    """
    Berechnet Statistiken aus den bisherigen Spielen eines Teams.
    """

    if len(history) == 0:
        return {
            "points_avg": 0,
            "goals_avg": 0,
            "conceded_avg": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "games": 0
        }

    points = [game["points"] for game in history]
    goals = [game["goals"] for game in history]
    conceded = [game["conceded"] for game in history]

    wins = sum(game["result"] == "W" for game in history)
    draws = sum(game["result"] == "D" for game in history)
    losses = sum(game["result"] == "L" for game in history)

    return {
        "points_avg": np.mean(points),
        "goals_avg": np.mean(goals),
        "conceded_avg": np.mean(conceded),
        "wins": wins,
        "draws": draws,
        "losses": losses,
        "games": len(history)
    }

## Vermeidung von Data Leakage

Beim Erzeugen der Features wird das aktuelle Spiel zunächst
vorhergesagt und erst anschließend in die Teamhistorie aufgenommen.

Dadurch können Informationen aus dem Ergebnis des aktuellen Spiels
nicht in dessen Features gelangen.

In [51]:
feature_rows = []

for _, row in df.iterrows():

    home_team = row["HomeTeam"]
    away_team = row["AwayTeam"]

    # Informationen VOR dem aktuellen Spiel
    home_stats = get_team_stats(team_history[home_team])
    away_stats = get_team_stats(team_history[away_team])

    # Statistiken aus den bisherigen Heimspielen
    home_home_stats = get_team_stats(home_history[home_team])

    # Statistiken aus den bisherigen Auswärtsspielen
    away_away_stats = get_team_stats(away_history[away_team])

    features = {
    "Date": row["Date"],
    "HomeTeam": home_team,
    "AwayTeam": away_team,

    # Gesamtform
    "home_points_avg": home_stats["points_avg"],
    "away_points_avg": away_stats["points_avg"],

    "home_goals_avg": home_stats["goals_avg"],
    "away_goals_avg": away_stats["goals_avg"],

    "home_conceded_avg": home_stats["conceded_avg"],
    "away_conceded_avg": away_stats["conceded_avg"],

    "home_wins": home_stats["wins"],
    "away_wins": away_stats["wins"],

    "home_draws": home_stats["draws"],
    "away_draws": away_stats["draws"],

    "home_losses": home_stats["losses"],
    "away_losses": away_stats["losses"],

    "home_games_history": home_stats["games"],
    "away_games_history": away_stats["games"],

    # Heim-/Auswärtsform
    "home_home_points_avg": home_home_stats["points_avg"],
    "away_away_points_avg": away_away_stats["points_avg"],

    "home_home_goals_avg": home_home_stats["goals_avg"],
    "away_away_goals_avg": away_away_stats["goals_avg"],

    "home_home_conceded_avg": home_home_stats["conceded_avg"],
    "away_away_conceded_avg": away_away_stats["conceded_avg"],

    "home_home_wins": home_home_stats["wins"],
    "away_away_wins": away_away_stats["wins"],

    "home_home_games": home_home_stats["games"],
    "away_away_games": away_away_stats["games"],

    # Zielvariable
    "FTR": row["FTR"]
}

    feature_rows.append(features)

    # Erst NACH der Feature-Erstellung
    # wird das aktuelle Spiel zur Historie hinzugefügt.

    if row["FTR"] == "H":
        home_points = 3
        away_points = 0
        home_result = "W"
        away_result = "L"

    elif row["FTR"] == "D":
        home_points = 1
        away_points = 1
        home_result = "D"
        away_result = "D"

    else:
        home_points = 0
        away_points = 3
        home_result = "L"
        away_result = "W"

    team_history[home_team].append({
        "points": home_points,
        "goals": row["FTHG"],
        "conceded": row["FTAG"],
        "result": home_result
    })

    team_history[away_team].append({
        "points": away_points,
        "goals": row["FTAG"],
        "conceded": row["FTHG"],
        "result": away_result
    })

    home_history[home_team].append({
    "points": home_points,
    "goals": row["FTHG"],
    "conceded": row["FTAG"],
    "result": home_result
    })

    away_history[away_team].append({
    "points": away_points,
    "goals": row["FTAG"],
    "conceded": row["FTHG"],
    "result": away_result
    })

In [52]:
features_df = pd.DataFrame(feature_rows)

In [53]:
features_df.head(10)

,Date,HomeTeam,AwayTeam,home_points_avg,away_points_avg,home_goals_avg,away_goals_avg,home_conceded_avg,away_conceded_avg,home_wins,...,away_away_points_avg,home_home_goals_avg,away_away_goals_avg,home_home_conceded_avg,away_away_conceded_avg,home_home_wins,away_away_wins,home_home_games,away_away_games,FTR
0,2010-08-20,Bayern Munich,Wolfsburg,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,H
1,2010-08-21,FC Koln,Kaiserslautern,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,A
2,2010-08-21,Freiburg,St Pauli,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,A
3,2010-08-21,Hamburg,Schalke 04,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,H
4,2010-08-21,Hannover,Ein Frankfurt,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,H
5,2010-08-21,Hoffenheim,Werder Bremen,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,H
6,2010-08-21,M'gladbach,Nurnberg,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,D
7,2010-08-22,Mainz,Stuttgart,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,H
8,2010-08-22,Dortmund,Leverkusen,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,A
9,2010-08-27,Kaiserslautern,Bayern Munich,3.0,3.0,3.0,2.0,1.0,1.0,1,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,H


In [54]:
features_df[
    features_df["HomeTeam"] == "Bayern Munich"
][
    [
        "Date",
        "AwayTeam",
        "home_points_avg",
        "home_goals_avg",
        "home_conceded_avg",
        "home_games_history"
    ]
].head(10)

,Date,AwayTeam,home_points_avg,home_goals_avg,home_conceded_avg,home_games_history
0,2010-08-20,Wolfsburg,0.000000,0.000000,0.0,0
19,2010-09-11,Werder Bremen,1.500000,1.000000,1.5,2
28,2010-09-18,FC Koln,1.333333,0.666667,1.0,3
50,2010-09-25,Mainz,1.600000,0.800000,0.8,5
65,2010-10-16,Hannover,1.000000,0.600000,1.0,5
81,2010-10-29,Freiburg,1.400000,1.200000,1.0,5
106,2010-11-14,Nurnberg,1.600000,2.000000,1.4,5
118,2010-11-27,Ein Frankfurt,1.800000,2.200000,1.2,5
136,2010-12-11,St Pauli,1.600000,2.200000,1.4,5
168,2011-01-22,Kaiserslautern,2.000000,2.600000,1.4,5


In [55]:
features_df[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "home_points_avg",
        "home_home_points_avg",
        "away_points_avg",
        "away_away_points_avg"
    ]
].head(15)

,Date,HomeTeam,AwayTeam,home_points_avg,home_home_points_avg,away_points_avg,away_away_points_avg
0,2010-08-20,Bayern Munich,Wolfsburg,0.0,0.0,0.0,0.0
1,2010-08-21,FC Koln,Kaiserslautern,0.0,0.0,0.0,0.0
2,2010-08-21,Freiburg,St Pauli,0.0,0.0,0.0,0.0
3,2010-08-21,Hamburg,Schalke 04,0.0,0.0,0.0,0.0
4,2010-08-21,Hannover,Ein Frankfurt,0.0,0.0,0.0,0.0
5,2010-08-21,Hoffenheim,Werder Bremen,0.0,0.0,0.0,0.0
6,2010-08-21,M'gladbach,Nurnberg,0.0,0.0,0.0,0.0
7,2010-08-22,Mainz,Stuttgart,0.0,0.0,0.0,0.0
8,2010-08-22,Dortmund,Leverkusen,0.0,0.0,0.0,0.0
9,2010-08-27,Kaiserslautern,Bayern Munich,3.0,0.0,3.0,0.0


In [56]:
features_df.isnull().sum()

Date                      0
HomeTeam                  0
AwayTeam                  0
home_points_avg           0
away_points_avg           0
home_goals_avg            0
away_goals_avg            0
home_conceded_avg         0
away_conceded_avg         0
home_wins                 0
away_wins                 0
home_draws                0
away_draws                0
home_losses               0
away_losses               0
home_games_history        0
away_games_history        0
home_home_points_avg      0
away_away_points_avg      0
home_home_goals_avg       0
away_away_goals_avg       0
home_home_conceded_avg    0
away_away_conceded_avg    0
home_home_wins            0
away_away_wins            0
home_home_games           0
away_away_games           0
FTR                       0
dtype: int64

In [57]:
features_df[
    [
        "home_points_avg",
        "away_points_avg",
        "home_goals_avg",
        "away_goals_avg"
    ]
].describe()

,home_points_avg,away_points_avg,home_goals_avg,away_goals_avg
count,4896.000000,4896.000000,4896.000000,4896.000000
mean,1.352866,1.397699,1.488460,1.527791
std,0.683100,0.685662,0.703508,0.721233
min,0.000000,0.000000,0.000000,0.000000
25%,0.800000,0.800000,1.000000,1.000000
50%,1.400000,1.400000,1.400000,1.400000
75%,1.800000,1.800000,1.800000,2.000000
max,3.000000,3.000000,5.000000,4.600000


In [58]:
features_df.to_csv(
    "../data/processed/bundesliga_features.csv",
    index=False
)

In [59]:
def expected_score(rating_a, rating_b):
    """
    Calculates the expected score of team A against team B.
    """
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(rating, expected, actual, k=20):
    """
    Updates an Elo rating after a match.
    """
    return rating + k * (actual - expected)

## Elo Rating

Das Elo-Rating dient als kompakte Schätzung der relativen Teamstärke.

Alle Teams starten mit einem Rating von 1500. Nach jedem Spiel wird
das Rating anhand des erwarteten und tatsächlichen Ergebnisses angepasst.

Wichtig ist, dass für die Feature-Erstellung immer das Elo-Rating
vor dem aktuellen Spiel verwendet wird.

In [60]:
elo_ratings = defaultdict(lambda: 1500)

In [61]:
elo_rows = []

for _, row in df.iterrows():

    home_team = row["HomeTeam"]
    away_team = row["AwayTeam"]

    # Elo ratings BEFORE the current match
    home_elo = elo_ratings[home_team]
    away_elo = elo_ratings[away_team]

    elo_rows.append({
        "Date": row["Date"],
        "HomeTeam": home_team,
        "AwayTeam": away_team,
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_difference": home_elo - away_elo
    })

    # Expected result
    expected_home = expected_score(home_elo, away_elo)
    expected_away = 1 - expected_home

    # Actual result
    if row["FTR"] == "H":
        actual_home = 1
        actual_away = 0

    elif row["FTR"] == "D":
        actual_home = 0.5
        actual_away = 0.5

    else:
        actual_home = 0
        actual_away = 1

    # Update ratings AFTER the match
    elo_ratings[home_team] = update_elo(
        home_elo,
        expected_home,
        actual_home
    )

    elo_ratings[away_team] = update_elo(
        away_elo,
        expected_away,
        actual_away
    )

In [62]:
elo_df = pd.DataFrame(elo_rows)

elo_df.head(10)

,Date,HomeTeam,AwayTeam,home_elo,away_elo,elo_difference
0,2010-08-20,Bayern Munich,Wolfsburg,1500.0,1500.0,0.0
1,2010-08-21,FC Koln,Kaiserslautern,1500.0,1500.0,0.0
2,2010-08-21,Freiburg,St Pauli,1500.0,1500.0,0.0
3,2010-08-21,Hamburg,Schalke 04,1500.0,1500.0,0.0
4,2010-08-21,Hannover,Ein Frankfurt,1500.0,1500.0,0.0
5,2010-08-21,Hoffenheim,Werder Bremen,1500.0,1500.0,0.0
6,2010-08-21,M'gladbach,Nurnberg,1500.0,1500.0,0.0
7,2010-08-22,Mainz,Stuttgart,1500.0,1500.0,0.0
8,2010-08-22,Dortmund,Leverkusen,1500.0,1500.0,0.0
9,2010-08-27,Kaiserslautern,Bayern Munich,1510.0,1510.0,0.0


In [63]:
features_df["home_elo"] = elo_df["home_elo"]
features_df["away_elo"] = elo_df["away_elo"]
features_df["elo_difference"] = elo_df["elo_difference"]

In [64]:
features_df["points_avg_diff"] = (
    features_df["home_points_avg"]
    - features_df["away_points_avg"]
)

features_df["goals_avg_diff"] = (
    features_df["home_goals_avg"]
    - features_df["away_goals_avg"]
)

features_df["conceded_avg_diff"] = (
    features_df["home_conceded_avg"]
    - features_df["away_conceded_avg"]
)

features_df["wins_diff"] = (
    features_df["home_wins"]
    - features_df["away_wins"]
)

In [65]:
features_df[
    [
        "home_elo",
        "away_elo",
        "elo_difference",
        "points_avg_diff",
        "goals_avg_diff",
        "conceded_avg_diff",
        "wins_diff"
    ]
].describe()

,home_elo,away_elo,elo_difference,points_avg_diff,goals_avg_diff,conceded_avg_diff,wins_diff
count,4896.000000,4896.000000,4896.000000,4896.000000,4896.000000,4896.000000,4896.000000
mean,1538.985541,1540.366154,-1.380613,-0.044833,-0.039331,0.038994,-0.080065
std,90.884637,90.791467,128.903698,0.961097,0.996321,0.888235,1.727185
min,1338.807768,1342.200076,-432.665872,-3.000000,-4.000000,-3.000000,-5.000000
25%,1477.125763,1478.644949,-75.303230,-0.800000,-0.600000,-0.600000,-1.000000
50%,1514.650304,1516.538501,-0.483762,0.000000,0.000000,0.000000,0.000000
75%,1583.862226,1584.038223,73.683804,0.600000,0.600000,0.600000,1.000000
max,1839.908687,1836.507676,419.493617,3.000000,3.800000,3.400000,5.000000


In [66]:
features_df[
    [
        "home_elo",
        "away_elo",
        "elo_difference",
        "points_avg_diff",
        "goals_avg_diff",
        "conceded_avg_diff",
        "wins_diff"
    ]
].isnull().sum()

home_elo             0
away_elo             0
elo_difference       0
points_avg_diff      0
goals_avg_diff       0
conceded_avg_diff    0
wins_diff            0
dtype: int64

In [67]:
features_df.to_csv(
    "../data/processed/bundesliga_features.csv",
    index=False
)